In [2]:
from openpyxl import load_workbook
import pandas as pd
import time
from datetime import datetime
import re
from collections import defaultdict
import os
from itertools import groupby

#### Project: These set of functions together take an excel file that tracks a certain cell line growth in our lab and converts it into a dataframe format.

In [3]:
### Step 1: Classify sheet as 'good' format or 'bad' format

In [4]:
### Step 2: For a good format sheet, On column b, look for the second date on that column. Look for uninterrupted chains of numbers where the chain length is divisble by 3 on that row with the date. 
##Once you've identified the chains, split that chain up. 
## If the chain is more than 1 cell from the date then don't proceed.
### Save the date, save the file name, get the passage from the sheet name. v1 you get from the first 3 in thatchain
### v2 you get from the second and v3 the third. The letter comes from counting the first multiple of the chain
#### Save that data in a dataframe.

In [3]:
def load_workbooks(folder_path): #Loads all workbooks in a folder 
    workbooks = []
    for filename in os.listdir(folder_path):
        if filename.endswith(('.xlsx', '.xlm')) and not filename.startswith('~$'):
            path = os.path.join(folder_path, filename)
            wb = load_workbook(path)
            workbooks.append({'filename': filename, 'path': path, 'workbook': wb})
    return workbooks

In [12]:
def group_numbers(nums):
    trimmed = nums[:-4]
    result = {}
    for i, val in enumerate(trimmed):
        letter = chr(ord('A') + i // 3)
        result.setdefault(letter, []).append(val)
    for k, v in result.items():
        while len(v) < 3:
            v.append(None)
    return result

In [30]:
gaussia_values = []
model_id = 686
passage = 0
workbooks = load_workbooks('./sgluc_files') # Loading all workbooks the sgluc folder into the 'workbooks' list 
rows = []
for item in workbooks:
    print(item['filename'])
    print(item['workbook'].sheetnames)
    gluc_file = item['filename']
    sheets = item['workbook'].sheetnames
    for name in sheets:                    
        ws = item['workbook'][name]
        for cell in ws["B"]:
            if isinstance(cell.value, datetime):   # Check for the first date in column B, most excel files have the dates in column B
                print(f"Datetime in {name}: {cell.value} at {cell.coordinate}")
                input_date = cell.value
                neighbor = ws.cell(row=cell.row, column=cell.column + 1) # Check if the next cell over is a date. If so continue.
                if isinstance(neighbor.value, (int, float)): 
                    offset = 1
                    while True: 
                        num = ws.cell(row=cell.row, column=cell.column + offset) # Get all sequential values into a list 
                        if isinstance(num.value, (int, float)):
                            gaussia_values.append(num.value)
                            offset += 1
                        else:
                            break
                    mice = group_numbers(gaussia_values)
                    for designation, values in mice.items():
                        value1, value2, value3 = values
                        if(value1 == None):
                            continue
                        rows.append({
                            'model': model_id,
                            'passage': passage,
                            'date': input_date,
                            'designation': designation,
                            'value1': value1,
                            'value2': value2,
                            'value3': value3,
                        })
                    gaussia_values = []
                    print("Sheet finished")

GS686-sGluc.xlsx
['GBX686P0', 'GBX686P1']
Datetime in GBX686P0: 2025-07-02 00:00:00 at B10
Datetime in GBX686P0: 2025-07-22 00:00:00 at B11
Sheet finished
Datetime in GBX686P0: 2025-08-06 00:00:00 at B12
Sheet finished
Datetime in GBX686P0: 2025-08-20 00:00:00 at B13
Sheet finished
Datetime in GBX686P0: 2025-09-03 00:00:00 at B14
Sheet finished
Datetime in GBX686P0: 2025-09-17 00:00:00 at B15
Sheet finished
Datetime in GBX686P0: 2025-10-01 00:00:00 at B16
Sheet finished
Datetime in GBX686P0: 2025-10-15 00:00:00 at B17
Sheet finished
Datetime in GBX686P0: 2025-10-22 00:00:00 at B18
Sheet finished
Datetime in GBX686P0: 2025-10-29 00:00:00 at B19
Sheet finished
Datetime in GBX686P0: 2025-11-05 00:00:00 at B20
Sheet finished
Datetime in GBX686P0: 2025-11-12 00:00:00 at B21
Sheet finished
Datetime in GBX686P0: 2025-11-19 00:00:00 at B22
Sheet finished
Datetime in GBX686P0: 2025-11-26 00:00:00 at B23
Sheet finished
Datetime in GBX686P0: 2025-12-03 00:00:00 at B24
Sheet finished
Datetime in G

In [8]:
print(mice)

{'A': [1675, 1610, 1850], 'B': [3025, 3250, 3435]}


In [31]:
df = pd.DataFrame(rows, columns=['model', 'passage', 'date', 'designation', 'value1', 'value2', 'value3'])

In [39]:
df_date = df[df['date'] == '2025-12-17']
df_date.head()

,model,passage,date,designation,value1,value2,value3
90,686,0,2025-12-17,A,132030,137405,142300
91,686,0,2025-12-17,B,2515,2765,2125
92,686,0,2025-12-17,C,78375,73915,80945
93,686,0,2025-12-17,D,136995,144920,170730
94,686,0,2025-12-17,E,11115,11815,12550


In [17]:
from datetime import datetime

gaussia_values = []
model_id = 686
passage = 0
workbooks = load_workbooks('./sgluc_files')  # Loading all workbooks in the sgluc folder into 'workbooks'
rows = []

for item in workbooks:
    print(item['filename'])
    print(item['workbook'].sheetnames)
    gluc_file = item['filename']
    sheets = item['workbook'].sheetnames
    for name in sheets:
        ws = item['workbook'][name]
        for cell in ws["B"]:
            if isinstance(cell.value, datetime):  # Check for the first date in column B
                print(f"Datetime in {name}: {cell.value} at {cell.coordinate}")
                input_date = cell.value
                row_num = cell.row

                # Check first 9 real-data columns (C onward, skipping A and the date in B)
                first_nine = [ws.cell(row=row_num, column=c).value for c in range(3, 12)]
                if not any(isinstance(v, (int, float)) for v in first_nine):
                    print(f"Row {row_num} has no numeric values in first 9 columns, skipping")
                    continue

                for col_index in range(1, ws.max_column + 1):
                    target_cell = ws.cell(row=row_num, column=col_index)
                    print(target_cell.value)

                    if col_index <= 2:   # skip column A and column B (the date column)
                        continue

                    next_cell = ws.cell(row=row_num, column=col_index + 1)
                    prev_cell = ws.cell(row=row_num, column=col_index - 1)
                    gaussia_values.append(target_cell.value)

                    if (target_cell.value is None
                            and next_cell.value is not None
                            and prev_cell.value is not None):
                        print("This is the dividing none")
                        break

                mice = group_numbers(gaussia_values)
                for designation, values in mice.items():
                    try:
                        value1, value2, value3 = values
                    except ValueError:
                        print(f"Skipping {designation}: expected 3 values, got {values}")
                        continue
                    else:
                        rows.append({
                            'model': model_id,
                            'passage': passage,
                            'date': input_date,
                            'designation': designation,
                            'value1': value1,
                            'value2': value2,
                            'value3': value3,
                        })

                gaussia_values = []
                break  # stop after first datetime found in this sheet
        print("Sheet finished")

GS686-sGluc.xlsx
['GBX686P0', 'GBX686P1']
Datetime in GBX686P0: 2025-07-02 00:00:00 at B10
Row 10 has no numeric values in first 9 columns, skipping
Datetime in GBX686P0: 2025-07-22 00:00:00 at B11
None
2025-07-22 00:00:00
2345
2985
2760
2970
3180
3510
3955
3915
3765
3480
3360
3315
3795
3945
4025
5650
5675
5520
1895
1910
1960
None
This is the dividing none
Sheet finished
Datetime in GBX686P1: 2026-05-21 00:00:00 at B10
Row 10 has no numeric values in first 9 columns, skipping
Datetime in GBX686P1: 2026-06-10 00:00:00 at B11
None
2026-06-10 00:00:00
1675
1610
1850
3025
3250
3435
1970
2045
2125
None
This is the dividing none
Sheet finished


In [18]:
print(mice)

{'A': [1675, 1610, 1850], 'B': [3025, 3250, 3435]}


In [8]:
df = pd.DataFrame(rows, columns=['model', 'passage', 'date', 'designation', 'value1', 'value2', 'value3'])

In [9]:
df.head(20)

,model,passage,date,designation,value1,value2,value3


In [33]:
for item in workbooks:
    print(item['filename'])
    print(item['workbook'].sheetnames)
    gluc_file = item['filename']
    sheets = item['workbook'].sheetnames
    for name in sheets:                    
        ws = item['workbook'][name]
        for cell in ws["B"]:
            for col_index in range(1, 30):
                target_cell = ws.cell(row=29, column=col_index)
                print(target_cell.value)
                next_cell = ws.cell(row=29, column=col_index+1)
                #print( "Next cell is " + str(next_cell))
                if(col_index == 1):
                    continue
                prev_cell = ws.cell(row=29, column=col_index-1)
                if((target_cell.value == None) & (next_cell.value != None) & (prev_cell.value != None)):
                        print("This is the dvividing None")
                        break

GS686-sGluc.xlsx
['GBX686P0', 'GBX686P1']
None
2026-01-21 00:00:00
None
None
None
8715
10365
10445
462295
515975
502920
1/20/2026
None
None
60825
64330
72665
15110
14425
15915
1835
1830
2080
None
This is the dvividing None
=_xlfn.DAYS(B29,$B$10)
=IF(ISBLANK(D29), "*", AVERAGE(C29:E29))
=IF(ISBLANK(G29), "*", AVERAGE(F29:H29))
=IF(ISBLANK(J29), "*", AVERAGE(I29:K29))
=IF(ISBLANK(M29), "*", AVERAGE(L29:N29))
None
2026-01-21 00:00:00
None
None
None
8715
10365
10445
462295
515975
502920
1/20/2026
None
None
60825
64330
72665
15110
14425
15915
1835
1830
2080
None
This is the dvividing None
=_xlfn.DAYS(B29,$B$10)
=IF(ISBLANK(D29), "*", AVERAGE(C29:E29))
=IF(ISBLANK(G29), "*", AVERAGE(F29:H29))
=IF(ISBLANK(J29), "*", AVERAGE(I29:K29))
=IF(ISBLANK(M29), "*", AVERAGE(L29:N29))
None
2026-01-21 00:00:00
None
None
None
8715
10365
10445
462295
515975
502920
1/20/2026
None
None
60825
64330
72665
15110
14425
15915
1835
1830
2080
None
This is the dvividing None
=_xlfn.DAYS(B29,$B$10)
=IF(ISBLANK(D29), 